# VoyageAI - Weather Agent

This notebook creates the Weather Agent for VoyageAI.

The Weather Agent is responsible for:
1. Reading the selected destination from the Destination Agent
2. Retrieving best-time-to-visit and season-related information from ChromaDB
3. Giving seasonal weather advice
4. Suggesting packing and activity tips
5. Returning structured weather guidance

This version does not use live weather APIs.
It gives seasonal guidance from the RAG knowledge base.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import json

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

e:\Agentic AI\VoyageAI Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


In [3]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "agents":
    PROJECT_ROOT = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

CHROMA_DIR = PROJECT_ROOT / "chroma_db"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Chroma DB folder:", CHROMA_DIR)

if not CHROMA_DIR.exists():
    raise FileNotFoundError("Chroma DB folder not found. Run 01_rag_ingestion.ipynb first.")

Current directory: e:\Agentic AI\VoyageAI Backend\notebooks\agents
Project root: e:\Agentic AI\VoyageAI Backend
Chroma DB folder: e:\Agentic AI\VoyageAI Backend\chroma_db


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2560.26it/s]


Embedding model loaded successfully.


In [5]:
vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model,
    collection_name="voyageai_travel_knowledge"
)

print("ChromaDB loaded successfully.")
print("Total vectors:", vector_store._collection.count())

ChromaDB loaded successfully.
Total vectors: 45


In [6]:
def get_available_destinations():
    collection_data = vector_store._collection.get(include=["metadatas"])

    destinations = set()

    for metadata in collection_data["metadatas"]:
        city = metadata.get("city")
        if city:
            destinations.add(city.lower().strip())

    return sorted(list(destinations))


available_destinations = get_available_destinations()

print("Available destinations in ChromaDB:")
print(available_destinations)

Available destinations in ChromaDB:
['agra', 'andaman', 'darjeeling', 'delhi', 'goa', 'jaipur', 'kashmir', 'kerala', 'ladakh', 'manali', 'mumbai', 'pondicherry', 'rishikesh', 'udaipur', 'varanasi']


In [7]:
def normalize_text(text: str):
    if not text:
        return ""

    return text.lower().strip()


def is_destination_available(destination_name: str):
    destination_name = normalize_text(destination_name)

    if not destination_name:
        return False

    for destination in available_destinations:
        if destination == destination_name:
            return True

        if destination in destination_name:
            return True

        if destination_name in destination:
            return True

    return False

In [8]:
DESTINATION_KEY_MAP = {
    "goa": "goa",
    "jaipur": "jaipur",
    "manali": "manali",
    "kerala": "kerala",
    "andaman": "andaman",
    "andaman and nicobar islands": "andaman",
    "udaipur": "udaipur",
    "rishikesh": "rishikesh",
    "varanasi": "varanasi",
    "ladakh": "ladakh",
    "kashmir": "kashmir",
    "darjeeling": "darjeeling",
    "pondicherry": "pondicherry",
    "puducherry": "pondicherry",
    "agra": "agra",
    "delhi": "delhi",
    "mumbai": "mumbai"
}


def normalize_destination_to_key(destination_name: str):
    if not destination_name:
        return None

    destination_name = destination_name.lower().strip()

    for name, key in DESTINATION_KEY_MAP.items():
        if name in destination_name:
            return key

    return destination_name.replace(" ", "_")

In [9]:
def get_weather_context(user_query: str, destination_result: dict, k: int = 4):
    destination = destination_result.get("recommended_destination")
    city_key = normalize_destination_to_key(destination)

    retrieval_query = f"""
    Destination: {destination}
    User request: {user_query}
    Need best time to visit, weather season, monsoon, summer, winter, snow, packing tips, travel risks, activity suitability.
    """

    try:
        docs = vector_store.similarity_search(
            retrieval_query,
            k=k,
            filter={"city": city_key}
        )

        if not docs:
            docs = vector_store.similarity_search(retrieval_query, k=k)

    except Exception as e:
        print("Filter search failed. Running normal similarity search.")
        print("Error:", e)
        docs = vector_store.similarity_search(retrieval_query, k=k)

    context_parts = []

    for i, doc in enumerate(docs, start=1):
        city = doc.metadata.get("city")
        source = doc.metadata.get("source")

        context = f"""
Context {i}
City: {city}
Source: {source}
Content:
{doc.page_content}
"""
        context_parts.append(context.strip())

    return "\n\n".join(context_parts)

In [10]:
user_query = "I want to visit Manali in December for snow and adventure"

destination_result = {
    "agent_name": "Destination Agent",
    "status": "success",
    "recommended_destination": "Manali",
    "reason": "Manali matches snow, mountains, adventure sports, cafes and scenic valleys.",
    "suitable_for": ["adventure seekers", "snow lovers", "nature lovers"],
    "suggested_duration": "4 to 6 days",
    "best_time_to_visit": "March to June for pleasant weather and December to February for snow",
    "confidence": "high"
}

weather_context = get_weather_context(user_query, destination_result)

print(weather_context)

Context 1
City: manali
Source: manali.txt
Content:
Best Time to Visit:
March to June is good for pleasant weather. December to February is good for snow experiences.

Ideal Duration:
4 to 6 days

Travel Tips:
- Rohtang Pass access depends on weather and government rules.
- Old Manali is better for cafes and backpacker stays.

Context 2
City: manali
Source: manali.txt
Content:
Destination: Manali
State/Region: Himachal Pradesh
Destination Type: Mountains, snow, adventure, honeymoon, backpacking, cafes

Overview:
Manali is a popular hill station known for mountains, snow activities, adventure sports, riverside stays, cafes, and scenic valleys.

Best For:
- Adventure seekers
- Honeymoon couples
- Backpackers
- Nature lovers
- Bikers
- Snow lovers

Context 3
City: manali
Source: manali.txt
Content:
Top Attractions:
- Solang Valley
- Hadimba Temple
- Old Manali
- Manu Temple
- Mall Road
- Vashisht Hot Springs
- Atal Tunnel
- Sissu
- Rohtang Pass

Food Recommendations:
- Trout fish
- Siddu
-

In [11]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

print("Groq LLM loaded successfully.")

Groq LLM loaded successfully.


In [12]:
weather_agent_prompt = ChatPromptTemplate.from_template("""
You are the Weather Agent of VoyageAI.

Your task is to give seasonal weather guidance for the selected destination based on:
1. User travel request
2. Destination Agent output
3. Retrieved destination knowledge

User Travel Request:
{user_query}

Destination Agent Output:
{destination_result}

Retrieved Travel Knowledge:
{context}

Return your answer strictly in valid JSON format.

JSON structure:
{{
  "agent_name": "Weather Agent",
  "status": "success",
  "destination": "destination name",
  "weather_advice_type": "seasonal_guidance_not_live_forecast",
  "requested_time_period": "month/season if mentioned by user, otherwise unknown",
  "best_time_to_visit": "best time from retrieved knowledge",
  "season_suitability": "high/medium/low/unknown",
  "weather_summary": "short seasonal weather summary",
  "activity_suitability": {{
    "sightseeing": "high/medium/low/unknown",
    "outdoor_activities": "high/medium/low/unknown",
    "water_activities": "high/medium/low/unknown",
    "snow_activities": "high/medium/low/unknown"
  }},
  "packing_tips": ["tip 1", "tip 2", "tip 3"],
  "weather_risks": ["risk 1", "risk 2"],
  "travel_advice": "practical advice based on the season",
  "limitations": "This version uses seasonal knowledge from the RAG database and does not use live weather forecast APIs.",
  "confidence": "high/medium/low"
}}

Rules:
- Use only the retrieved travel knowledge.
- Do not invent live temperature, rainfall, wind speed, AQI, or exact weather forecast.
- Do not claim to know current weather.
- If the user asks for live/current weather, clearly mention the limitation.
- If the user mentions a month or season, compare it with the best time to visit from the retrieved knowledge.
- Do not recommend weather details outside the selected destination context.
- Do not add markdown.
- Do not wrap JSON in ```json.
""")

In [13]:
weather_agent_chain = weather_agent_prompt | llm | StrOutputParser()

print("Weather Agent chain created successfully.")

Weather Agent chain created successfully.


In [14]:
def parse_json_response(response: str):
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        try:
            start = response.find("{")
            end = response.rfind("}") + 1
            json_str = response[start:end]
            return json.loads(json_str)
        except Exception:
            print("Failed to parse JSON. Raw response:")
            print(response)
            return {
                "error": "Failed to parse JSON",
                "raw_response": response
            }

In [15]:
def run_weather_agent(user_query: str, destination_result: dict, k: int = 4):
    destination_status = destination_result.get("status")
    destination = destination_result.get("recommended_destination")

    if destination_status != "success":
        return {
            "agent_name": "Weather Agent",
            "status": "skipped",
            "destination": None,
            "message": "Weather Agent skipped because Destination Agent did not return a valid destination.",
            "reason": destination_result.get("reason"),
            "confidence": "low"
        }

    if not destination:
        return {
            "agent_name": "Weather Agent",
            "status": "no_destination_found",
            "destination": None,
            "message": "Weather Agent cannot run because no destination was provided.",
            "confidence": "low"
        }

    if not is_destination_available(destination):
        return {
            "agent_name": "Weather Agent",
            "status": "out_of_knowledge_base",
            "destination": destination,
            "message": f"{destination} is not available in the current VoyageAI knowledge base.",
            "available_destinations": available_destinations,
            "confidence": "low"
        }

    context = get_weather_context(
        user_query=user_query,
        destination_result=destination_result,
        k=k
    )

    response = weather_agent_chain.invoke({
        "user_query": user_query,
        "destination_result": json.dumps(destination_result, indent=2),
        "context": context
    })

    parsed_response = parse_json_response(response)

    parsed_response["status"] = parsed_response.get("status", "success")

    return parsed_response

In [16]:

user_query = "I want to visit Manali in December for snow and adventure"

destination_result = {
    "agent_name": "Destination Agent",
    "status": "success",
    "recommended_destination": "Manali",
    "reason": "Manali matches snow, mountains, adventure sports, cafes and scenic valleys.",
    "suitable_for": ["adventure seekers", "snow lovers", "nature lovers"],
    "suggested_duration": "4 to 6 days",
    "best_time_to_visit": "March to June for pleasant weather and December to February for snow",
    "confidence": "high"
}

weather_result = run_weather_agent(user_query, destination_result)

print(json.dumps(weather_result, indent=2))

{
  "agent_name": "Weather Agent",
  "status": "success",
  "destination": "Manali",
  "weather_advice_type": "seasonal_guidance_not_live_forecast",
  "requested_time_period": "December",
  "best_time_to_visit": "December to February for snow",
  "season_suitability": "high",
  "weather_summary": "Manali experiences cold weather with snowfall in December, making it ideal for snow activities and adventure sports.",
  "activity_suitability": {
    "sightseeing": "high",
    "outdoor_activities": "high",
    "water_activities": "low",
    "snow_activities": "high"
  },
  "packing_tips": [
    "Pack warm clothing, including heavy jackets, gloves, and scarves.",
    "Bring waterproof gear to protect against snow and rain.",
    "Wear comfortable shoes for outdoor activities."
  ],
  "weather_risks": [
    "Rohtang Pass access may be restricted due to weather conditions and government rules.",
    "Old Manali may experience heavy snowfall, making it difficult to access some areas."
  ],
  "t

In [17]:
user_query = "What is the current weather in Goa right now?"

destination_result = {
    "agent_name": "Destination Agent",
    "status": "success",
    "recommended_destination": "Goa",
    "reason": "Goa matches beach and coastal travel.",
    "suitable_for": ["beach lovers", "relaxed vacation seekers"],
    "suggested_duration": "3 to 5 days",
    "best_time_to_visit": "November to February",
    "confidence": "high"
}

weather_result = run_weather_agent(user_query, destination_result)

print(json.dumps(weather_result, indent=2))

{
  "agent_name": "Weather Agent",
  "status": "success",
  "destination": "Goa",
  "weather_advice_type": "seasonal_guidance_not_live_forecast",
  "requested_time_period": "unknown",
  "best_time_to_visit": "November to February",
  "season_suitability": "high",
  "weather_summary": "Goa experiences a tropical monsoon climate with warm temperatures and high humidity throughout the year. The best time to visit is during the winter months (November to February) when the weather is cooler and drier.",
  "activity_suitability": {
    "sightseeing": "high",
    "outdoor_activities": "high",
    "water_activities": "high",
    "snow_activities": "unknown"
  },
  "packing_tips": [
    "Pack light and breathable clothing for warm weather.",
    "Bring sunscreen and a hat for sun protection.",
    "Wear comfortable shoes for walking and outdoor activities.",
    "Pack a light jacket or sweater for cooler evenings."
  ],
  "weather_risks": [
    "Heat and humidity can be intense during the summ